In [1]:
%pip install -q google-genai sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [14]:
import getpass
from pathlib import Path

import faiss
import numpy as np
import pandas as pd

from google import genai
from sentence_transformers import SentenceTransformer

from IPython.display import display, Markdown

In [3]:
def find_repository_root(start_path=Path.cwd()):
    for folder in [start_path, *start_path.parents]:
        if (folder / ".git").exists():
            return folder
    
    raise FileNotFoundError(
        "Could not locate the Git repository."
    )


REPO_ROOT = find_repository_root()
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"

chunks_path = (
    PROCESSED_DATA_DIR / "indexed_chunks.jsonl"
)

index_path = (
    PROCESSED_DATA_DIR /
    "creditlens_bge_small.faiss"
)

print("Chunks:", chunks_path)
print("Index:", index_path)

Chunks: D:\analytics\A_Python_Code\creditlens-rag\data\processed\indexed_chunks.jsonl
Index: D:\analytics\A_Python_Code\creditlens-rag\data\processed\creditlens_bge_small.faiss


In [4]:
chunks_df = pd.read_json(
    chunks_path,
    lines=True
)

faiss_index = faiss.read_index(
    str(index_path)
)

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cpu"
)

print("Chunks loaded:", len(chunks_df))
print("Vectors loaded:", faiss_index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chunks loaded: 3913
Vectors loaded: 3913


In [5]:
assert len(chunks_df) == faiss_index.ntotal

print("Chunk-to-vector mapping verified.")

Chunk-to-vector mapping verified.


In [6]:
QUERY_PREFIX = (
    "Represent this sentence for searching "
    "relevant passages: "
)


def semantic_search(
    question,
    company=None,
    reporting_year=None,
    top_k=5
):
    query_embedding = embedding_model.encode(
        [QUERY_PREFIX + question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")
    
    candidate_count = min(
        max(top_k * 20, 100),
        faiss_index.ntotal
    )
    
    scores, ids = faiss_index.search(
        query_embedding,
        candidate_count
    )
    
    results = (
        chunks_df
        .iloc[ids[0]]
        .copy()
        .reset_index(drop=True)
    )
    
    results.insert(
        0,
        "similarity_score",
        scores[0]
    )
    
    if company is not None:
        results = results[
            results["company"].str.lower()
            == company.lower()
        ]
    
    if reporting_year is not None:
        results = results[
            results["reporting_year"].astype(str)
            == str(reporting_year)
        ]
    
    return (
        results
        .head(top_k)
        .reset_index(drop=True)
    )

In [7]:
gemini_api_key = getpass.getpass(
    "Enter your Gemini API key: "
)

gemini_client = genai.Client(
    api_key=gemini_api_key
)

print("Gemini client created.")

Enter your Gemini API key:  ········


Gemini client created.


In [8]:
test_interaction = (
    gemini_client.interactions.create(
        model="gemini-3.6-flash",
        input=(
            "Reply with exactly: "
            "CreditLens connection successful."
        ),
        store=False
    )
)

print(test_interaction.output_text)

CreditLens connection successful.


In [9]:
def build_context(search_results):
    context_blocks = []
    
    for source_number, row in search_results.iterrows():
        source_label = f"Source {source_number + 1}"
        
        context_block = f"""
[{source_label}]
Company: {row['company']}
Reporting year: {row['reporting_year']}
Filing type: {row['form']}
Chunk ID: {row['chunk_id']}
SEC URL: {row['source_url']}

Passage:
{row['text']}
"""
        
        context_blocks.append(context_block.strip())
    
    return "\n\n".join(context_blocks)

In [10]:
def build_rag_prompt(question, search_results):
    context = build_context(search_results)
    
    prompt = f"""
You are a corporate credit-risk research assistant.

Use only the SEC filing passages provided below.

Rules:
1. Do not use outside knowledge.
2. Cite every material claim using labels such as [Source 1].
3. Do not invent financial figures or conclusions.
4. If the supplied evidence is insufficient, clearly say so.
5. Distinguish disclosed risks from events that have already occurred.
6. Give a concise credit-analyst-style answer.
7. End with a short "Evidence limitations" statement.

SEC FILING EVIDENCE

{context}

QUESTION

Based only on the preceding evidence, answer this question:

{question}
"""
    
    return prompt.strip()

In [11]:
def generate_rag_answer(
    question,
    company,
    reporting_year,
    top_k=5
):
    search_results = semantic_search(
        question=question,
        company=company,
        reporting_year=reporting_year,
        top_k=top_k
    )
    
    if search_results.empty:
        return {
            "answer": (
                "No relevant evidence was retrieved."
            ),
            "sources": search_results
        }
    
    prompt = build_rag_prompt(
        question,
        search_results
    )
    
    interaction = (
        gemini_client.interactions.create(
            model="gemini-3.6-flash",
            input=prompt,
            generation_config={
                "thinking_level": "low"
            },
            store=False
        )
    )
    
    return {
        "answer": interaction.output_text,
        "sources": search_results
    }

In [15]:
question = (
    "What factors could negatively affect "
    "Ford's liquidity?"
)

rag_result = generate_rag_answer(
    question=question,
    company="Ford",
    reporting_year=2025,
    top_k=5
)

#print(rag_result["answer"])
display(
    Markdown(rag_result["answer"])
)

Based on the provided SEC filing passages, the factors that could negatively affect Ford’s and Ford Credit’s liquidity include:

### Disclosed Risk Factors That Could Affect Liquidity

* **Funding and Debt Capital Market Disruption:** Ford Credit’s ability to maintain liquidity can be impacted by prolonged disruptions in debt and securitization markets, global capital markets volatility, lower market capacity for Ford- and Ford Credit-sponsored investments, and general demand shifts for its offered securities [Source 1].
* **Credit Rating Downgrades and Access to Financing:** Credit ratings assigned to Ford and Ford Credit, as well as downgrades, market disruptions, regulatory requirements, or asset portfolio performance, could impair access to debt, securitization, or derivative markets at competitive rates or in sufficient amounts [Source 1, Source 3].
* **Asset-Backed Financing and Hedging Limitations:** Reduced ability to continue funding through asset-backed financing structures, poor performance of underlying assets in these structures, an inability to obtain committed asset-backed/credit facilities, or an inability to secure hedging instruments can negatively affect liquidity [Source 1].
* **High Interest Rate Environment:** Elevated interest rates increase the cost of capital for capital-intensive businesses like Ford and can reduce Ford Credit’s financing margins by affecting its ability to source funding and offer competitive financing [Source 2].
* **Pension and OPEB Plan Obligations:** Significant liabilities associated with defined benefit pension plans and postretirement obligations (OPEB) could require additional cash contributions that impair liquidity [Source 4]. If cash flows are insufficient to meet these obligations, Ford could be forced to delay investments, suspend dividends, seek capital, or restructure debt [Source 4].
* **EV Adoption and Regulatory Compliance Pressures:** Slower-than-anticipated consumer demand for electric vehicles (EVs) or rapid shifts in consumer demand away from larger, more profitable internal combustion vehicles could force production curtailments or necessitate purchasing third-party compliance credits, impacting operational cash flow and revenues [Source 2, Source 5].
* **Credit and Residual Value Losses:** Higher-than-expected credit losses, lower residual values, or higher return volumes on leased vehicles at Ford Credit, as well as fluctuations/unrealized losses in the market value of investments, present risk to financial results [Source 2, Source 3].

***

### Evidence Limitations
The provided passages consist primarily of risk factor disclosures and do not contain quantitative current-period liquidity metrics (such as exact cash balances, available credit facility amounts, or specific debt maturity schedules) or past historical cash-flow figures.

In [13]:
for index, row in rag_result["sources"].iterrows():
    print(f"Source {index + 1}")
    print("Company:", row["company"])
    print("Year:", row["reporting_year"])
    print("Score:", round(row["similarity_score"], 4))
    print("Chunk:", row["chunk_id"])
    print("URL:", row["source_url"])
    print()

Source 1
Company: Ford
Year: 2025
Score: 0.8165
Chunk: F_2025_10K_chunk_0315
URL: https://www.sec.gov/Archives/edgar/data/37996/000003799626000015/f-20251231.htm

Source 2
Company: Ford
Year: 2025
Score: 0.7884
Chunk: F_2025_10K_chunk_0173
URL: https://www.sec.gov/Archives/edgar/data/37996/000003799626000015/f-20251231.htm

Source 3
Company: Ford
Year: 2025
Score: 0.7879
Chunk: F_2025_10K_chunk_0328
URL: https://www.sec.gov/Archives/edgar/data/37996/000003799626000015/f-20251231.htm

Source 4
Company: Ford
Year: 2025
Score: 0.7815
Chunk: F_2025_10K_chunk_0185
URL: https://www.sec.gov/Archives/edgar/data/37996/000003799626000015/f-20251231.htm

Source 5
Company: Ford
Year: 2025
Score: 0.7733
Chunk: F_2025_10K_chunk_0243
URL: https://www.sec.gov/Archives/edgar/data/37996/000003799626000015/f-20251231.htm



In [16]:
source_table = rag_result["sources"][
    [
        "similarity_score",
        "company",
        "reporting_year",
        "chunk_id",
        "source_url"
    ]
].copy()

source_table.insert(
    0,
    "source_label",
    [
        f"Source {number + 1}"
        for number in range(len(source_table))
    ]
)

source_table

,source_label,similarity_score,company,reporting_year,chunk_id,source_url
0,Source 1,0.816513,Ford,2025,F_2025_10K_chunk_0315,https://www.sec.gov/Archives/edgar/data/37996/...
1,Source 2,0.788357,Ford,2025,F_2025_10K_chunk_0173,https://www.sec.gov/Archives/edgar/data/37996/...
2,Source 3,0.787853,Ford,2025,F_2025_10K_chunk_0328,https://www.sec.gov/Archives/edgar/data/37996/...
3,Source 4,0.781454,Ford,2025,F_2025_10K_chunk_0185,https://www.sec.gov/Archives/edgar/data/37996/...
4,Source 5,0.773271,Ford,2025,F_2025_10K_chunk_0243,https://www.sec.gov/Archives/edgar/data/37996/...


In [17]:
unsupported_question = (
    "What exact credit rating will Ford "
    "receive next year?"
)

unsupported_result = generate_rag_answer(
    question=unsupported_question,
    company="Ford",
    reporting_year=2025,
    top_k=5
)

display(
    Markdown(unsupported_result["answer"])
)

**Credit Analyst Analysis**

Based on the provided SEC filing passages, it is **impossible to determine** what exact credit rating Ford will receive next year [Source 1, Source 2, Source 3, Source 4, Source 5]. 

The provided excerpts contain historical financial details, liquidity metrics, debt facility agreements, and internal risk ratings that Ford Credit assigns to its dealership portfolio [Source 1, Source 3, Source 4, Source 5]. However, none of the passages disclose, project, or discuss future credit ratings assigned to Ford Motor Company by external credit rating agencies (e.g., S&P, Moody's, Fitch).

---

### Evidence Limitations
* **Missing Information:** The provided SEC filing excerpts contain no data regarding external corporate credit ratings, agency rating outlooks, or credit rating projections for future years.